# Wishart: быстрый перебор радиуса плотности k (без повторного сжатия)

Этот блокнот работает **только с результатами уже завершённого эксперимента** на ConceptNet-100k. Он читает одну доверенную контрольную точку (около 19 МБ в сжатом виде) и четыре сохранённых таблицы типов, но **не сканирует граф заново, не выполняет VF2, MDL или contraction**.

Мы переигрываем Wishart для k = 1, 2, 3, 4, 6, 12, сохраняя исходные типы, частоты, typed-WL, significance=0.7 и минимальные размеры семейств. Ближайшие соседи считаются **один раз на уровне**, а затем из полного упорядоченного списка берутся первые k. Для каждого k делаем 20 повторов на случайных 80% типов, пересчитывая соседей **среди оставшихся типов**.

Результат: таблица числа семейств, шума, размера крупнейшего семейства, доли нулевых радиусов, устойчивости (ARI) и сравнение k=12 с сохранённой исходной кластеризацией. Все результаты пишутся в отдельный каталог Drive, а не в исходный run.

**Импорт пакета:** после установки editable-пакета блокнот явно добавляет `repo/src` в `sys.path`, поскольку уже запущенный Python kernel может не прочитать новый `.pth` до перезапуска. Ячейка 3 проверяет импорт заранее.

**Безопасность:** pickle допускается загружать только из собственного, доверенного запуска. Контрольная точка проверяется по SHA-256 и метаданным перед загрузкой. Не указывайте чужие или неизвестные checkpoint-файлы.

**Ограничение интерпретации:** в текущем алгоритме Wishart-семейства не управляют MDL-выбором, поэтому этот блокнот проверяет качество *кластеризации*, а не улучшение сжатия.

In [ ]:
# 1. Все параметры находятся здесь.
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime, timezone

DRIVE_ROOT = Path('/content/drive/MyDrive/SemanticMap/colab/wishart')
SOURCE_RUN_NAME = 'wishart-dictionary-100k-gpu-gpu'
K_VALUES = (1, 2, 3, 4, 6, 12)
SUBSET_FRACTION = 0.80
SUBSET_REPEATS = 20
SEED = 1729

# auto = CUDA, если доступна; для маленьких популяций CPU тоже подходит.
REPLAY_DEVICE = 'auto'
GPU_BATCH_SIZE = 128
CPU_WORKERS = 1  # несколько сотен типов: запуск процессов обычно дороже вычисления
ANALYSIS_NAME = 'knn-replay-k1-k12'

REPO_URL = 'https://github.com/SemanticMap/semgraphex.git'
BRANCH = 'feature/wishart-knn-replay-notebook'
ANALYSIS_CODE_SHA = 'adee6194e8ec8d183af1ec75244c551d6c06a20f'
EXPECTED_SOURCE_CODE_SHA = '95bab776beaa9e1c438518ffc0c9ac103ab4a529'
REPO_DIR = Path('/content/semgraphex-knn-replay')
SCRATCH = Path('/content/semmap-knn-replay')
SOURCE_RUN = DRIVE_ROOT / 'runs' / SOURCE_RUN_NAME
OUTPUT_DRIVE = DRIVE_ROOT / 'analysis' / SOURCE_RUN_NAME / ANALYSIS_NAME

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(name, '1')
SCRATCH.mkdir(parents=True, exist_ok=True)
print({
    'python': sys.version.split()[0], 'cpu_count': os.cpu_count(),
    'scratch_free_gb': round(shutil.disk_usage(SCRATCH).free / 1e9, 1),
    'source': str(SOURCE_RUN), 'analysis_output': str(OUTPUT_DRIVE),
    'k_grid': K_VALUES,
})

In [ ]:
# 2. Монтируем Google Drive и проверяем исходный запуск — только чтение.
from google.colab import drive
drive.mount('/content/drive')

if not (SOURCE_RUN / 'COMPLETED').is_file():
    raise FileNotFoundError(
        f'Не найден завершённый запуск: {SOURCE_RUN}. '
        'Измените SOURCE_RUN_NAME на название своей папки runs/.'
    )
source_input = json.loads((SOURCE_RUN / 'input.json').read_text(encoding='utf-8'))
source_hierarchy = json.loads((SOURCE_RUN / 'hierarchy.json').read_text(encoding='utf-8'))
if source_input['code_revision'] != EXPECTED_SOURCE_CODE_SHA:
    raise ValueError(
        'Этот блокнот проверен для конкретного исходного Git-коммита. '
        f"Получен {source_input['code_revision']}; ожидается {EXPECTED_SOURCE_CODE_SHA}."
    )
if source_hierarchy['options']['metric'] != 'typed_wl':
    raise ValueError('Быстрый replay ожидает исходный typed_wl, а не другую метрику.')
print({
    'source_code': source_input['code_revision'],
    'source_device': source_input['device']['selected'],
    'original_k': source_hierarchy['options']['k_neighbors'],
    'completed_transitions': len(source_hierarchy['transitions']),
    'stop_reason': source_hierarchy['stop_reason'],
})
if source_hierarchy['options']['k_neighbors'] != 12:
    raise ValueError('Для сравнения с исходными метками ожидается k_neighbors=12.')

In [ ]:
# 3. Клонируем код анализа и устанавливаем зависимости без обновления всего Colab.
# Для закрытого GitHub-репозитория нужен Colab Secret GITHUB_TOKEN с read-доступом.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
public_probe = subprocess.run(
    ['git', 'ls-remote', REPO_URL, BRANCH],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
).returncode == 0
env = os.environ.copy()
askpass_path = None
if not public_probe:
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception as error:
        raise RuntimeError(
            'Добавьте GITHUB_TOKEN в Colab Secrets, разрешите доступ этому notebook '
            'и повторите ячейку.'
        ) from error
    if not token:
        raise RuntimeError('GITHUB_TOKEN пуст.')
    fd, raw_path = tempfile.mkstemp(prefix='semmap-git-askpass-', suffix='.sh')
    os.close(fd)
    askpass_path = Path(raw_path)
    askpass_path.write_text(
        '#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*Password*) echo "$GITHUB_TOKEN" ;;\nesac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    env['GIT_ASKPASS'] = str(askpass_path)
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['GITHUB_TOKEN'] = token
try:
    subprocess.run(
        ['git', 'clone', '--depth', '30', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        env=env, check=True,
    )
finally:
    if askpass_path is not None:
        askpass_path.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN', None)

subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', ANALYSIS_CODE_SHA], check=True)
ancestry = subprocess.run([
    'git', '-C', str(REPO_DIR), 'merge-base', '--is-ancestor',
    EXPECTED_SOURCE_CODE_SHA, ANALYSIS_CODE_SHA,
]).returncode
if ancestry != 0:
    raise RuntimeError('Исходный код не является предком кода анализа: воспроизводимость не подтверждена.')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-c', str(REPO_DIR / 'requirements/constraints-colab.txt'),
    '-e', str(REPO_DIR) + '[wishart,notebook]',
], check=True)
# Editable-install из subprocess создаёт .pth, который уже работающий kernel
# может не прочитать до перезапуска. Явно добавляем src-layout в sys.path.
REPO_SRC = REPO_DIR / 'src'
PACKAGE_INIT = REPO_SRC / 'semmap_haken' / '__init__.py'
if not PACKAGE_INIT.is_file():
    raise RuntimeError(
        f'Не найден исходный пакет: {PACKAGE_INIT}. '
        'Проверьте git clone и ANALYSIS_CODE_SHA в этой ячейке.'
    )
repo_src_str = str(REPO_SRC.resolve())
if repo_src_str not in sys.path:
    sys.path.insert(0, repo_src_str)
import importlib
importlib.invalidate_caches()
try:
    import semmap_haken
    from semmap_haken.wishart_resume import load_latest_checkpoint
    from semmap_haken.wishart_knn_replay import sweep_level
except ModuleNotFoundError as error:
    raise RuntimeError(
        'Не удалось импортировать semmap_haken после установки. '
        'Выполните ячейку 3 целиком, проверьте вывод pip и Python kernel.'
    ) from error
actual_package = Path(semmap_haken.__file__).resolve()
if not actual_package.is_relative_to(REPO_SRC.resolve()):
    raise RuntimeError(
        f'Python импортировал другой semmap_haken: {actual_package}. '
        'Перезапустите kernel и выполните блокнот сверху вниз.'
    )
os.chdir(REPO_DIR)
print('Пакет успешно импортирован:', actual_package)
print('Python kernel:', sys.executable)
print('Код анализа:', ANALYSIS_CODE_SHA)
print('Исходный код:', EXPECTED_SOURCE_CODE_SHA)

In [ ]:
# 4. Скопируем ТОЛЬКО один последний проверенный checkpoint в быстрый /content.
# Никакого повторного сканирования или построения графа не будет.
if 'REPO_SRC' not in globals() or not (REPO_SRC / 'semmap_haken' / '__init__.py').is_file():
    raise RuntimeError('Сначала выполните ячейку 3 (clone, pip install, проверка импорта).')
from semmap_haken.wishart_resume import load_latest_checkpoint

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

LOCAL_RUN = SCRATCH / 'source_checkpoint'
LOCAL_CHECKPOINTS = LOCAL_RUN / 'checkpoints'
LOCAL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
drive_checkpoints = sorted(
    (p for p in (SOURCE_RUN / 'checkpoints').glob('level_*') if p.is_dir()),
    reverse=True,
)
if not drive_checkpoints:
    raise FileNotFoundError('В исходном запуске нет checkpoints/level_*.')
chosen_manifest = None
for source_checkpoint in drive_checkpoints:
    try:
        manifest = json.loads((source_checkpoint / 'manifest.json').read_text(encoding='utf-8'))
    except (OSError, ValueError) as error:
        print('Пропускаю checkpoint с нечитаемым manifest:', source_checkpoint.name, error)
        continue
    for field, expected in (
        ('config_sha256', source_input['config_sha256']),
        ('input_sha256', source_input['input_sha256']),
        ('code_revision', source_input['code_revision']),
    ):
        if manifest.get(field) != expected:
            raise RuntimeError(f'{source_checkpoint.name}: не совпадает {field}')
    if manifest.get('state_file') != 'state.pkl.gz':
        raise RuntimeError('Неожиданный формат checkpoint.')
    local_checkpoint = LOCAL_CHECKPOINTS / source_checkpoint.name
    local_checkpoint.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_checkpoint / 'manifest.json', local_checkpoint / 'manifest.json')
    shutil.copy2(source_checkpoint / 'state.pkl.gz', local_checkpoint / 'state.pkl.gz')
    local_payload = local_checkpoint / 'state.pkl.gz'
    if local_payload.stat().st_size != manifest['state_size'] or sha256_file(local_payload) != manifest['state_sha256']:
        print('Checkpoint не прошёл SHA-256:', source_checkpoint.name)
        shutil.rmtree(local_checkpoint)
        continue
    chosen_manifest = manifest
    print('Проверен checkpoint:', source_checkpoint.name, 'МБ:', round(local_payload.stat().st_size / 1e6, 1))
    break
if chosen_manifest is None:
    raise RuntimeError('Не удалось найти checkpoint с корректной контрольной суммой.')

# pickle.load вызывается ТОЛЬКО после проверки manifest SHA-256. Источник обязан быть доверенным.
state = load_latest_checkpoint(
    LOCAL_RUN,
    config_hash=source_input['config_sha256'],
    input_hash=source_input['input_sha256'],
    code_revision=source_input['code_revision'],
)
if state is None:
    raise RuntimeError('Восстановление состояния не удалось.')
print({
    'completed_levels': state['next_level'],
    'exact_types_in_checkpoint': len(state['dictionary'].types),
    'source_snapshot': chosen_manifest['next_level'],
})

In [ ]:
# 5. Для каждого завершённого перехода восстановим ТОЧНЫЕ частоты типов
# из checkpoint. Сверим множество типов с исходным type_labels.npz.
import numpy as np
import pandas as pd
from semmap_haken.wishart_config import WishartOptions, DictionaryOptions
from semmap_haken.wishart_metrics import build_neighbor_graph
from semmap_haken.wishart_knn_replay import sweep_level

options = WishartOptions(**source_hierarchy['options'])
dictionary_options = DictionaryOptions(**source_hierarchy['dictionary_options'])
dictionary = state['dictionary']
count_by_type_and_level = dictionary._candidate_level_counts

levels = []
for transition in source_hierarchy['transitions']:
    level = int(transition['source_level'])
    target = int(transition['target_level'])
    if target != level + 1 or target > int(state['next_level']):
        continue
    source_labels = SOURCE_RUN / f'transition_{level:03d}_{target:03d}' / 'type_labels.npz'
    if not source_labels.is_file():
        raise FileNotFoundError(source_labels)
    local_labels = SCRATCH / f'original_type_labels_{level:03d}.npz'
    shutil.copy2(source_labels, local_labels)
    with np.load(local_labels, allow_pickle=False) as old:
        old_type_ids = tuple(str(s) for s in old['type_ids'])
        old_labels = old['labels'].astype(np.int64)

    counts = {
        type_id: int(frequency)
        for (type_id, source_level), frequency in count_by_type_and_level.items()
        if int(source_level) == level and int(frequency) >= dictionary_options.min_support
    }
    type_ids = tuple(sorted(counts))
    if type_ids != old_type_ids:
        raise RuntimeError(
            f'На уровне {level} типы в checkpoint не совпадают с исходным type_labels.npz. '
            'Результаты могут принадлежать разным запускам.'
        )
    if len(type_ids) <= max(K_VALUES):
        raise RuntimeError(f'На уровне {level} слишком мало типов для k={max(K_VALUES)}.')
    levels.append({
        'level': level, 'type_ids': type_ids,
        'weights': np.asarray([counts[type_id] for type_id in type_ids], dtype=float),
        'original_labels': old_labels,
    })
if not levels:
    raise RuntimeError('Нет завершённых переходов для replay.')
print(pd.DataFrame([
    {'level': item['level'], 'supported_types': len(item['type_ids']),
     'frequency_mass': int(item['weights'].sum())}
    for item in levels
]).to_string(index=False))

In [ ]:
# 6. Один полный kNN-граф на уровне, затем дешёвый replay для всех k.
# Список соседей содержит ВСЕ n-1 других типов: это необходимо для
# корректного пересчёта kNN в каждой случайной 80%-ной подвыборке.
all_rows = []
all_label_arrays = {}
backends = {}
for level_data in levels:
    level = level_data['level']
    ids = level_data['type_ids']
    representatives = tuple(dictionary.representative(type_id) for type_id in ids)
    neighbors = build_neighbor_graph(
        representatives,
        metric=options.metric,
        k=len(ids) - 1,
        wl_iterations=options.wl_iterations,
        feature_dim=options.feature_dim,
        graphlet_size=options.graphlet_size,
        graphlet_samples=options.graphlet_samples,
        transport_rank=options.transport_rank,
        transport_max_candidates=options.transport_max_candidates,
        fgw_alpha=options.fgw_alpha,
        relation_js_block_size=options.relation_js_block_size,
        seed=options.random_seed + 3001 * level,
        device=REPLAY_DEVICE,
        gpu_batch_size=GPU_BATCH_SIZE,
        min_gpu_types=1,
        cpu_workers=CPU_WORKERS,
    )
    backends[level] = dict(neighbors.metadata)
    rows, labels_by_k = sweep_level(
        level=level,
        full_indices=neighbors.indices,
        full_distances=neighbors.distances,
        weights=(
            level_data['weights']
            if options.density_weight == 'occurrence_frequency'
            else np.ones(len(ids), dtype=float)
        ),
        significance=options.significance,
        min_cluster_size=options.min_cluster_size,
        min_cluster_mass=(
            options.min_cluster_mass
            if options.density_weight == 'occurrence_frequency' else None
        ),
        k_values=K_VALUES,
        original_labels=level_data['original_labels'],
        subset_fraction=SUBSET_FRACTION,
        subset_repeats=SUBSET_REPEATS,
        seed=SEED,
    )
    all_rows.extend(rows)
    all_label_arrays[f'level_{level:03d}_type_ids'] = np.asarray(ids, dtype=str)
    for k, labels in labels_by_k.items():
        all_label_arrays[f'level_{level:03d}_k_{k:02d}_labels'] = labels
    baseline = next(row for row in rows if row['k'] == 12)
    print(
        f"Уровень {level}: типы={len(ids)}, backend={backends[level]['backend']}, "
        f"k=12 vs исходный ARI={baseline['baseline_ari']:.4f}, "
        f"точные метки={baseline['baseline_labels_exact']}"
    )
    if baseline['baseline_ari'] < 0.999:
        print('ВНИМАНИЕ: исходная кластеризация не воспроизведена точно; '
              'проверьте GPU/CPU backend, версии библиотек и численные совпадения.')

table = pd.DataFrame(all_rows).sort_values(['level', 'k']).reset_index(drop=True)
display(table[[
    'level', 'k', 'type_count', 'family_count', 'noise_fraction',
    'largest_family_fraction', 'zero_kth_radius_fraction',
    'stability_ari_median', 'baseline_ari'
]])
# Если получается одно большое семейство, ARI может быть тривиально высоким:
# обязательно смотрите family_count, largest_family_fraction и noise_fraction.

In [ ]:
# 7. Сводка и графики. Не назначаем «победителя» только по одной метрике.
import matplotlib.pyplot as plt

summary = table.groupby('k', as_index=False).agg(
    levels=('level', 'count'),
    median_families=('family_count', 'median'),
    median_noise=('noise_fraction', 'median'),
    median_largest_family=('largest_family_fraction', 'median'),
    median_stability_ari=('stability_ari_median', 'median'),
    median_zero_radius=('zero_kth_radius_fraction', 'median'),
)
display(summary)

plots = [
    ('family_count', 'Количество семейств', 'Семейств'),
    ('noise_fraction', 'Доля типов вне семейств', 'Доля шума'),
    ('largest_family_fraction', 'Доля типов в крупнейшем семействе', 'Доля'),
    ('stability_ari_median', 'Устойчивость на 80% типов', 'Медиана ARI'),
    ('zero_kth_radius_fraction', 'Нулевой k-радиус', 'Доля'),
]
FIGURE_DIR = SCRATCH / 'plots'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
for col, title, ylabel in plots:
    fig, ax = plt.subplots(figsize=(7, 4))
    for level, group in table.groupby('level'):
        ax.plot(group['k'], group[col], marker='o', label=f'уровень {level}')
    ax.set(xlabel='k_neighbors', ylabel=ylabel, title=title)
    ax.set_xticks(K_VALUES)
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f'{col}.png', dpi=150)
    plt.show()
    plt.close(fig)

if (table.query('k == 12')['baseline_ari'] < 0.999).any():
    print('Внимание: обнаружено несовпадение replay k=12 с исходным запуском. '
          'Выводы о стабильности сравнивайте с указанием backend.')
print('Обратите внимание: k=1 — граничный случай; высокое ARI при одном семействе не означает хорошее разделение.')

In [ ]:
# 8. Сохраним результаты в отдельной папке анализа. Исходный запуск не меняем.
# Копируем готовые локальные файлы; COMPLETED записывается последним.
LOCAL_OUTPUT = SCRATCH / 'analysis_output'
if LOCAL_OUTPUT.exists():
    shutil.rmtree(LOCAL_OUTPUT)
LOCAL_OUTPUT.mkdir(parents=True)
table.to_csv(LOCAL_OUTPUT / 'sweep_by_level.csv', index=False)
summary.to_csv(LOCAL_OUTPUT / 'summary_by_k.csv', index=False)
np.savez_compressed(LOCAL_OUTPUT / 'family_labels_by_level_k.npz', **all_label_arrays)
shutil.copytree(FIGURE_DIR, LOCAL_OUTPUT / 'plots', dirs_exist_ok=True)

analysis_manifest = {
    'analysis_kind': 'wishart_offline_knn_replay_no_coarsening',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'source_run_name': SOURCE_RUN_NAME,
    'source_input_sha256': source_input['input_sha256'],
    'source_config_sha256': source_input['config_sha256'],
    'source_git_revision': source_input['code_revision'],
    'analysis_git_revision': ANALYSIS_CODE_SHA,
    'checkpoint_level': int(state['next_level']),
    'checkpoint_state_sha256': chosen_manifest['state_sha256'],
    'levels': [row['level'] for row in levels],
    'k_values': list(K_VALUES),
    'subset_fraction': SUBSET_FRACTION,
    'subset_repeats': SUBSET_REPEATS,
    'seed': SEED,
    'wishart_options': source_hierarchy['options'],
    'dictionary_options': source_hierarchy['dictionary_options'],
    'device_requested': REPLAY_DEVICE,
    'nearest_neighbor_backends': backends,
    'baseline_note': 'k=12 is compared to saved original labels; GPU float32 and CPU float64 may differ near ties.',
    'stability_note': 'ARI compares full-label restriction to reclustering an 80% subset; one-cluster agreement can be trivial.',
    'scope_note': 'No graph extraction, exact-type discovery, MDL selection, Huffman recoding or contraction was executed.',
}
(LOCAL_OUTPUT / 'manifest.json').write_text(
    json.dumps(analysis_manifest, ensure_ascii=False, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)
OUTPUT_DRIVE.mkdir(parents=True, exist_ok=True)
(OUTPUT_DRIVE / 'COMPLETED').unlink(missing_ok=True)
for local_file in sorted(LOCAL_OUTPUT.rglob('*')):
    if local_file.is_file():
        target = OUTPUT_DRIVE / local_file.relative_to(LOCAL_OUTPUT)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_file, target)
        if local_file.stat().st_size != target.stat().st_size:
            raise IOError(f'Ошибка копирования: {local_file}')
(OUTPUT_DRIVE / 'COMPLETED').write_text('complete\n', encoding='utf-8')
print('Сохранено:', OUTPUT_DRIVE)
print('Таблица по уровням:', OUTPUT_DRIVE / 'sweep_by_level.csv')
print('Сводка по k:', OUTPUT_DRIVE / 'summary_by_k.csv')
print('Метки всех семейств:', OUTPUT_DRIVE / 'family_labels_by_level_k.npz')

## Как интерпретировать

Сначала посмотрите, воспроизвёлся ли контроль k=12: baseline_ari должен быть близок к 1. Затем сравните количество семейств, долю шума, крупнейшее семейство и устойчивость ARI. Высокая устойчивость при одном огромном семействе — не основание выбирать k. Нулевые радиусы особенно важны при k=1: совпадающие typed-WL признаки делают плотность резко выраженной.

Если несколько значений k выглядят устойчивыми, минимальным рабочим кандидатом для следующего полного эксперимента будет k=2 или k=3. Выбор следует делать по таблице, а не предполагать результат заранее. Для проверки реального сжатия потребуется **другая** серия: добавить использование Wishart-семейств в кодирование и MDL, а затем повторить сжатие.

Эта тетрадь не модифицирует исходную папку runs/ и не перезаписывает исходные контрольные точки.